# Cuaderno 8 — Primeros gráficos con ggplot2

**Descripción y Visualización de Datos — UAI 2026 — Clase 8**

Llevamos seis clases haciendo tablas. Hoy, por primera vez, las dibujamos.

La herramienta es `ggplot2`. Tiene fama de difícil, pero todo gráfico se escribe
con las mismas tres piezas, siempre en el mismo orden:

| Pieza | Responde a | Ejemplo |
|---|---|---|
| `ggplot(curso, ...)` | ¿con **qué datos**? | la base del curso |
| `aes(...)` | ¿qué columna va en **cada eje**? | `aes(x = transporte)` |
| `geom_...()` | ¿con **qué forma** la dibujo? | `geom_bar()` = barras |

Y una cuarta para terminarlo: `labs()`, que le pone título, ejes y fuente.

Nada más. Hoy aprendes cuatro formas: barras, histograma, columnas y puntos.

## Parte 0 — Punto de partida

Abrimos `dplyr` como siempre y, además, `ggplot2`. La tercera línea sólo fija el
tamaño de los gráficos en pantalla.

In [ ]:
library(dplyr)
library(ggplot2)
options(repr.plot.width = 8, repr.plot.height = 5)

curso <- read.csv("https://raw.githubusercontent.com/naimbro/naimbro.github.io/main/materiales/2026_descripcion_visualizacion_datos/datos/encuesta_curso.csv")

nrow(curso)

Antes de graficar, la limpieza de la clase 5: tres columnas que llegaron como
texto pasan a número. Esta vez **guardamos** el resultado con `curso <-`, para
que las columnas nuevas queden en la base y no haya que repetir esto en cada
celda.

El aviso en rojo (*NAs introduced by coercion*) ya lo conoces: son el `"10 min"`,
el `"3 horas"` y el `"5 horas"`.

In [ ]:
curso <- curso %>%
  mutate(minutos_num = as.numeric(minutos_viaje),
         sueno_num   = as.numeric(horas_sueno),
         redes_num   = as.numeric(horas_redes))

## Parte 1 — Un gráfico en tres pasos

Vamos a construir el primero pieza por pieza, para ver qué hace cada una.

**Paso 1: los datos.**

In [ ]:

ggplot(curso)

Un rectángulo gris vacío. `ggplot()` ya sabe **qué datos** usar, pero todavía no
sabe qué dibujar.

**Paso 2: los ejes.** `aes()` dice qué columna va en el eje x.

In [ ]:

ggplot(curso, aes(x = transporte))

Ahora hay un eje con las tres categorías, pero sigue sin barras: sabe **dónde**,
no sabe **con qué forma**.

**Paso 3: la forma.** Le **sumamos** una geometría con `+`.

In [ ]:
ggplot(curso, aes(x = transporte)) +
  geom_bar()

Ése es tu primer gráfico: 12 en auto, 6 en metro, 15 en micro.

Compáralo con la tabla que ya sabes hacer:

In [ ]:

curso %>% count(transporte)

Los mismos tres números. **`geom_bar()` hace el `count()` por ti** y lo dibuja.

> **El error más común de hoy:** dentro de un gráfico las piezas se **suman** con
> `+`, no se encadenan con `%>%`. El `%>%` sirve para la tabla; el `+`, para el
> dibujo. Si te equivocas, R te lo dice: *Did you use `%>%` or `|>` instead of
> `+`?* Y el `+` va **al final** de la línea, nunca al comienzo de la siguiente.

**1.** Un gráfico de barras de `sistema_operativo`.

In [ ]:

# Tu código acá

**2.** Lo mismo con `experiencia`. Las etiquetas son largas y se van a pisar.
Arréglalo cambiando `x` por `y` dentro de `aes()`: las barras quedan acostadas.

In [ ]:

# Tu código acá

**3.** Barras de `dominio`, acostadas. ¿Qué categorías te parecen raras?
*(Pista: son las mismas rarezas que ya habías visto con `count()`.)*

In [ ]:

# Tu código acá

## Parte 2 — Distribución: ¿cómo se reparten los números?

Las barras sirven para **categorías**. Para una columna de **números**, como las
horas en redes sociales, se usa un **histograma**: agrupa los valores en tramos y
cuenta cuántas personas caen en cada uno.

In [ ]:
ggplot(curso, aes(x = redes_num)) +
  geom_histogram(binwidth = 1)

`binwidth = 1` es el **ancho de cada tramo**: una hora. Casi todo el curso está
entre 2 y 5 horas diarias, y hay una cola que llega hasta 10.

Aparece un aviso: *Removed 1 row containing non-finite...* Es el `NA` del
`"5 horas"`. Como en la clase 5, no es un error: R te avisa que dejó a una
persona fuera del dibujo.

> **Por esto limpiamos en la Parte 0.** Prueba el mismo gráfico con `horas_redes`,
> la columna original: no corre. R responde *`stat_bin()` requires a continuous x
> aesthetic*: un histograma necesita números, y esa columna es texto.

**4.** Un histograma de `minutos_num`, con tramos de 15 minutos.

In [ ]:

# Tu código acá

**5.** Un histograma de `edad`, con tramos de 1 año. ¿Qué ves en el gráfico
que un promedio no te mostraría?

In [ ]:

# Tu código acá

## Parte 3 — Comparación: un número por grupo

`geom_bar()` cuenta filas. Pero muchas veces lo que queremos dibujar **no es un
conteo**, sino un número que ya calculamos: un promedio por grupo, por ejemplo.

Primero la tabla, con la receta de la clase 7. La guardamos con un nombre,
`viajes`, para poder graficarla:

In [ ]:
viajes <- curso %>%
  group_by(transporte) %>%
  summarise(minutos = mean(minutos_num, na.rm = TRUE))

viajes

Y ahora el gráfico. Esta vez `aes()` lleva **dos** columnas: la categoría en `x`
y el número en `y`. La forma es `geom_col()`:

In [ ]:
ggplot(viajes, aes(x = transporte, y = minutos)) +
  geom_col()

Fíjate que el primer argumento ya no es `curso`, sino `viajes`: **se grafica la
tabla que acabas de hacer**.

| | Qué le das | Qué dibuja |
|---|---|---|
| `geom_bar()` | sólo `x` | cuenta las filas de cada categoría |
| `geom_col()` | `x` **y** `y` | la altura que tú calculaste |

**6.** Las horas de sueño promedio (`sueno_num`) por `transporte`: primero la
tabla con `group_by()` y `summarise()`, después el gráfico con `geom_col()`.

In [ ]:

# Tu código acá

## Parte 4 — Relación: dos números a la vez

¿Los que pasan más horas en redes duermen menos? Para cruzar **dos columnas
numéricas** se usan puntos: cada persona es un punto.

In [ ]:
ggplot(curso, aes(x = redes_num, y = sueno_num)) +
  geom_point()

Cuenta los puntos. Son **17**, y en el curso hay 32 personas con las dos
respuestas.

No es un error de R: cinco personas respondieron 4 horas de redes y 6 de sueño,
y otras cinco, 5 y 6. Cada grupo de cinco quedó **en un solo punto**, uno encima
del otro. Con respuestas en números redondos, pasa todo el tiempo.

`geom_count()` lo arregla: dibuja un punto más grande donde hay más personas.

In [ ]:
ggplot(curso, aes(x = redes_num, y = sueno_num)) +
  geom_count()

Ahora sí se ve la tendencia, aunque es débil: los que duermen 8 horas o más están
a la izquierda, con pocas horas de redes.

`aes()` no sólo sirve para los ejes. También puede decidir el **color**:

In [ ]:
ggplot(curso, aes(x = redes_num, y = sueno_num, color = transporte)) +
  geom_count()

**7.** Un gráfico de puntos de `minutos_num` (eje x) contra `sueno_num` (eje y),
con el color según `transporte`. ¿Duermen menos los que viajan más?

In [ ]:

# Tu código acá

## Parte 5 — Terminarlo: títulos, ejes, unidades y fuente

Todo lo que hiciste hasta acá es un gráfico **de trabajo**: sirve para que tú
mires los datos. Para que lo lea otra persona, que no ve tu código, le faltan
cuatro cosas. Se agregan con `labs()`, sumado con `+` como cualquier pieza:

In [ ]:
ggplot(viajes, aes(x = transporte, y = minutos)) +
  geom_col() +
  labs(title    = "En auto se llega en la mitad del tiempo",
       subtitle = "Minutos de viaje, promedio por medio de transporte",
       x        = "Medio de transporte",
       y        = "Minutos (promedio)",
       caption  = "Fuente: encuesta del curso DVD 2026, 33 estudiantes") +
  theme_minimal()

La última línea, `theme_minimal()`, cambia el **tema**: saca el fondo gris. Es
opcional.

El control antes de mostrar un gráfico:

| | Pregunta |
|---|---|
| **Título** | ¿Dice lo que se ve, con un número o una comparación? *(Como el titular de la clase 6.)* |
| **Ejes** | ¿Cada eje dice qué mide y **en qué unidad**? |
| **Fuente** | ¿Dice de dónde vienen los datos y cuántas personas son? |

**8.** Termina el histograma de horas en redes de la Parte 2 con `labs()`:
título con un hallazgo, los dos ejes con su unidad y la fuente.

In [ ]:

# Tu código acá

**9. Tu gráfico.** Elige una columna de la base de tu grupo (la del Sprint 2) y
haz un gráfico terminado: la forma correcta (barras, histograma, columnas o
puntos) y `labs()` completo. Es el primer borrador del Sprint 3.

In [ ]:

# Tu código acá

### Desafío opcional

En el gráfico del ejercicio 3, las barras salen en orden alfabético. Arma primero
la tabla con `count(dominio)`, y grafícala con `geom_col()` usando
`aes(x = n, y = reorder(dominio, n))`. ¿Qué hace `reorder()`?

In [ ]:

# Tu código acá

## Antes de irte

**Archivo → Guardar** (Ctrl+S).

### Lo que aprendiste hoy

| Para qué | Cómo se escribe |
|---|---|
| Empezar un gráfico | `ggplot(curso, aes(x = transporte))` |
| Contar categorías | `+ geom_bar()` |
| Ver cómo se reparte un número | `+ geom_histogram(binwidth = 1)` |
| Dibujar un número que ya calculaste | `ggplot(viajes, aes(x = transporte, y = minutos)) + geom_col()` |
| Cruzar dos números | `+ geom_point()` o `+ geom_count()` |
| Colorear por grupo | `aes(..., color = transporte)` |
| Barras acostadas | `aes(y = experiencia)` en vez de `aes(x = ...)` |
| Terminarlo | `+ labs(title = , x = , y = , caption = )` |
| Sacar el fondo gris | `+ theme_minimal()` |

### Las tres ideas

1. **Todo gráfico son tres piezas: datos, `aes()` y `geom`.** Lo demás es
   terminación. Si un gráfico no sale, revisa cuál de las tres falta.
2. **La forma depende de la pregunta.** Categorías: barras. Cómo se reparte un
   número: histograma. Un número por grupo: columnas. Dos números: puntos.
3. **Un gráfico sin título, unidades y fuente no está terminado.** Quien lo lee
   no ve tu código; sólo ve el dibujo.

### Para la próxima clase

Elegir el gráfico correcto: qué forma usar según la pregunta, y los errores
que hacen que un gráfico mienta.